In [4]:
# ============================================
# ENHANCED PREDICTION WITH CLINICAL ADVICE
# ============================================

import joblib
import pandas as pd
import numpy as np
from datetime import datetime

# Load models
model_path = r'C:\Users\Sammy\Desktop\Methodology'
model = joblib.load(f'{model_path}/heart_disease_model.pkl')
scaler = joblib.load(f'{model_path}/scaler.pkl')
imputer = joblib.load(f'{model_path}/imputer.pkl')
feature_names = joblib.load(f'{model_path}/feature_names.pkl')

def get_clinical_recommendations(patient_data, prediction, probability, risk_factors):
    """
    Generate neutral, clinical recommendations based on assessment results
    Suitable for both healthcare providers and patients
    """
    
    recommendations = {
        'clinical_actions': [],
        'diagnostic_tests': [],
        'lifestyle_factors': [],
        'monitoring_parameters': [],
        'referral_suggestions': [],
        'urgency_level': '',
        'follow_up_timeline': ''
    }
    
    # URGENCY AND FOLLOW-UP based on risk level
    if prediction == 1 or probability >= 0.7:
        recommendations['urgency_level'] = 'HIGH - Prompt clinical evaluation indicated'
        recommendations['follow_up_timeline'] = 'Follow-up within 1-2 weeks'
        recommendations['clinical_actions'].append("Complete cardiac evaluation including history and physical examination")
        recommendations['clinical_actions'].append("Review cardiovascular risk factors and consider preventive interventions")
        recommendations['referral_suggestions'].append("Cardiology consultation for specialized assessment")
    elif probability >= 0.3:
        recommendations['urgency_level'] = 'MODERATE - Clinical assessment recommended'
        recommendations['follow_up_timeline'] = 'Follow-up within 1-3 months'
        recommendations['clinical_actions'].append("Primary care evaluation for cardiovascular risk assessment")
        recommendations['referral_suggestions'].append("Consider cardiology referral based on clinical judgment")
    else:
        recommendations['urgency_level'] = 'LOW - Routine monitoring'
        recommendations['follow_up_timeline'] = 'Regular follow-up as per standard guidelines'
        recommendations['clinical_actions'].append("Continue routine preventive care")
    
    # DIAGNOSTIC TESTING recommendations based on risk factors
    if 'High Cholesterol' in risk_factors:
        recommendations['diagnostic_tests'].append("Lipid panel for comprehensive cholesterol profile")
        recommendations['monitoring_parameters'].append("Monitor LDL cholesterol levels")
    
    if 'High Blood Pressure' in risk_factors:
        recommendations['diagnostic_tests'].append("Ambulatory blood pressure monitoring if indicated")
        recommendations['monitoring_parameters'].append("Regular blood pressure monitoring")
    
    if 'Significant ST Depression' in risk_factors or 'Exercise-Induced Angina' in risk_factors:
        recommendations['diagnostic_tests'].append("Exercise stress test or pharmacological stress test")
        recommendations['diagnostic_tests'].append("Consider stress echocardiography or nuclear imaging")
    
    if 'Multiple Vessel Disease' in risk_factors:
        recommendations['diagnostic_tests'].append("Coronary angiography for anatomical assessment")
        recommendations['diagnostic_tests'].append("CT coronary angiogram as alternative non-invasive option")
    
    if 'Reversible Thalassemia Defect' in risk_factors:
        recommendations['diagnostic_tests'].append("Myocardial perfusion scan to assess ischemia")
        recommendations['diagnostic_tests'].append("Consider coronary flow reserve measurement")
    
    if prediction == 1 or probability >= 0.5:
        recommendations['diagnostic_tests'].append("Resting 12-lead ECG for baseline assessment")
        recommendations['diagnostic_tests'].append("Transthoracic echocardiogram for structural evaluation")
    
    # LIFESTYLE FACTORS - neutral, evidence-based recommendations
    if 'High Cholesterol' in risk_factors:
        recommendations['lifestyle_factors'].append("Dietary modification: reduce saturated fat and increase fiber intake")
        recommendations['lifestyle_factors'].append("Consider Mediterranean or DASH dietary pattern")
    
    if 'High Blood Pressure' in risk_factors:
        recommendations['lifestyle_factors'].append("Sodium restriction: target <2300mg sodium per day")
        recommendations['lifestyle_factors'].append("Potassium-rich foods if no contraindications")
    
    if 'High Blood Sugar' in risk_factors:
        recommendations['lifestyle_factors'].append("Carbohydrate monitoring and glycemic control")
        recommendations['lifestyle_factors'].append("Regular blood glucose monitoring as indicated")
    
    # General lifestyle recommendations (always include)
    recommendations['lifestyle_factors'].extend([
        "Regular physical activity: 150 minutes moderate intensity per week",
        "Smoking cessation counseling if applicable",
        "Alcohol consumption within recommended limits",
        "Stress management techniques as appropriate",
        "Maintain healthy body weight (BMI 18.5-24.9 kg/m²)"
    ])
    
    # Exercise recommendations based on risk level
    if prediction == 1 or probability >= 0.7:
        recommendations['lifestyle_factors'].insert(0, "Medically supervised exercise program initially")
        recommendations['lifestyle_factors'].insert(1, "Gradual increase in physical activity based on tolerance")
    elif probability >= 0.3:
        recommendations['lifestyle_factors'].insert(0, "Gradual increase in physical activity with symptom monitoring")
    
    # MONITORING PARAMETERS
    if prediction == 1 or probability >= 0.7:
        recommendations['monitoring_parameters'].extend([
            "Daily blood pressure recording if hypertensive",
            "Symptom diary: chest discomfort, shortness of breath, palpitations",
            "Medication adherence monitoring",
            "Regular weight measurements"
        ])
    else:
        recommendations['monitoring_parameters'].extend([
            "Periodic blood pressure checks",
            "Annual lipid profile as per guidelines",
            "Symptom awareness and reporting"
        ])
    
    # Remove duplicate monitoring parameters
    recommendations['monitoring_parameters'] = list(set(recommendations['monitoring_parameters']))
    
    # PHARMACOLOGICAL CONSIDERATIONS (neutral language)
    if prediction == 1 or probability >= 0.7:
        recommendations['clinical_actions'].extend([
            "Discuss antiplatelet therapy based on risk-benefit assessment",
            "Evaluate need for statin therapy for lipid management",
            "Consider blood pressure lowering agents if hypertensive",
            "Review medication indications and contraindications"
        ])
    
    return recommendations

def identify_risk_factors(patient_data):
    """Identify specific risk factors from patient data"""
    risk_factors = []
    
    # Age-related factors
    if patient_data['Age'] > 65:
        risk_factors.append('Advanced Age (>65 years)')
    elif patient_data['Age'] > 55:
        risk_factors.append('Age >55 years')
    
    # Cholesterol
    if patient_data['Cholesterol'] > 240:
        risk_factors.append('High Cholesterol (>240 mg/dL)')
    elif patient_data['Cholesterol'] > 200:
        risk_factors.append('Borderline Cholesterol (200-240 mg/dL)')
    
    # Blood Pressure
    if patient_data['Resting Blood Pressure'] > 140:
        risk_factors.append('Elevated Blood Pressure (>140 mmHg)')
    elif patient_data['Resting Blood Pressure'] > 120:
        risk_factors.append('Pre-hypertension (120-140 mmHg)')
    
    # Blood Sugar
    if patient_data['Fasting Blood Sugar'] == 1:
        risk_factors.append('Impaired Fasting Glucose (>120 mg/dL)')
    
    # Cardiac markers
    if patient_data['Max Heart Rate'] < 120:
        risk_factors.append('Reduced Exercise Capacity (HR <120 bpm)')
    
    if patient_data['ST Depression'] > 2.0:
        risk_factors.append('Significant ST Depression (>2.0mm)')
    elif patient_data['ST Depression'] > 1.0:
        risk_factors.append('Mild ST Depression (1.0-2.0mm)')
    
    # Anatomical factors
    if patient_data['Number of Major Vessels'] >= 2:
        risk_factors.append(f'Multi-vessel Disease ({patient_data["Number of Major Vessels"]} vessels)')
    elif patient_data['Number of Major Vessels'] == 1:
        risk_factors.append('Single-vessel Disease')
    
    # Thalassemia
    if patient_data['Thalassemia'] == 7:
        risk_factors.append('Reversible Perfusion Defect')
    elif patient_data['Thalassemia'] == 6:
        risk_factors.append('Fixed Perfusion Defect')
    
    # Symptoms
    if patient_data['Chest Pain Type'] == 1:
        risk_factors.append('Typical Angina Symptoms')
    elif patient_data['Chest Pain Type'] == 2:
        risk_factors.append('Atypical Angina Symptoms')
    
    if patient_data['Exercise-Induced Angina'] == 1:
        risk_factors.append('Exercise-induced Symptoms')
    
    # ECG findings
    if patient_data['Slope of ST Segment'] == 3:
        risk_factors.append('Downsloping ST Segment')
    
    return risk_factors

def predict_with_recommendations(patient_data):
    """
    Enhanced prediction function that returns results with clinical recommendations
    """
    # Convert to DataFrame
    patient_df = pd.DataFrame([patient_data])
    
    # Add engineered features
    patient_df['Cholesterol_Risk'] = (patient_df['Cholesterol'] > 240).astype(int)
    patient_df['BP_Risk'] = (patient_df['Resting Blood Pressure'] > 140).astype(int)
    patient_df['HR_Risk'] = (patient_df['Max Heart Rate'] < 120).astype(int)
    
    # Reorder columns
    patient_df = patient_df[feature_names]
    
    # Transform
    patient_imputed = imputer.transform(patient_df)
    patient_scaled = scaler.transform(patient_df)
    
    # Predict
    prediction = model.predict(patient_scaled)[0]
    probability = model.predict_proba(patient_scaled)[0][1]
    
    # Identify risk factors
    risk_factors = identify_risk_factors(patient_data)
    
    # Get clinical recommendations
    recommendations = get_clinical_recommendations(patient_data, prediction, probability, risk_factors)
    
    return {
        'has_heart_disease': bool(prediction),
        'probability': float(probability),
        'risk_level': 'HIGH' if probability >= 0.7 else 'MODERATE' if probability >= 0.3 else 'LOW',
        'risk_factors': risk_factors,
        'recommendations': recommendations
    }

def print_clinical_report(patient_data, result):
    """
    Print a professionally formatted clinical report with neutral recommendations
    Suitable for both clinical and patient use
    """
    print("\n" + "="*80)
    print("CARDIOVASCULAR RISK ASSESSMENT REPORT")
    print("="*80)
    print(f"Assessment Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print(f"Report ID: {datetime.now().strftime('%Y%m%d%H%M%S')}")
    print("="*80)
    
    # Patient Demographics
    print("\nPATIENT INFORMATION")
    print("-"*80)
    print(f"  Age: {patient_data['Age']} years")
    print(f"  Sex: {'Male' if patient_data['Sex'] == 1 else 'Female'}")
    print(f"  Clinical Presentation: {'Symptomatic' if patient_data['Chest Pain Type'] in [1,2] else 'Asymptomatic'}")
    
    # Key Clinical Parameters
    print("\nKEY CLINICAL PARAMETERS")
    print("-"*80)
    print(f"  • Total Cholesterol: {patient_data['Cholesterol']} mg/dL")
    print(f"  • Resting Blood Pressure: {patient_data['Resting Blood Pressure']} mmHg")
    print(f"  • Fasting Glucose: {'>120 mg/dL' if patient_data['Fasting Blood Sugar'] == 1 else '<120 mg/dL'}")
    print(f"  • Maximum Heart Rate: {patient_data['Max Heart Rate']} bpm")
    print(f"  • ST Depression: {patient_data['ST Depression']} mm")
    print(f"  • Major Vessels Affected: {patient_data['Number of Major Vessels']}")
    
    # Assessment Result
    print("\nASSESSMENT RESULT")
    print("-"*80)
    if result['has_heart_disease']:
        print(f"  Finding: POSITIVE for coronary artery disease indicators")
        print(f"  Probability: {result['probability']:.2%}")
        print(f"  Risk Stratification: {result['risk_level']}")
    else:
        print(f"  Finding: NEGATIVE for coronary artery disease indicators")
        print(f"  Probability of being disease-free: {(1-result['probability']):.2%}")
        print(f"  Risk Stratification: {result['risk_level']}")
    
    # Identified Risk Factors
    if result['risk_factors']:
        print("\nIDENTIFIED RISK FACTORS")
        print("-"*80)
        for i, factor in enumerate(result['risk_factors'], 1):
            print(f"  {i}. {factor}")
    
    # Clinical Recommendations
    print("\nCLINICAL RECOMMENDATIONS")
    print("-"*80)
    print(f"  Priority Level: {result['recommendations']['urgency_level']}")
    print(f"  Suggested Timeline: {result['recommendations']['follow_up_timeline']}")
    
    if result['recommendations']['clinical_actions']:
        print("\n  Clinical Actions:")
        for action in result['recommendations']['clinical_actions']:
            print(f"    • {action}")
    
    if result['recommendations']['diagnostic_tests']:
        print("\n  Diagnostic Considerations:")
        for test in result['recommendations']['diagnostic_tests']:
            print(f"    • {test}")
    
    if result['recommendations']['referral_suggestions']:
        print("\n  Referral Considerations:")
        for referral in result['recommendations']['referral_suggestions']:
            print(f"    • {referral}")
    
    if result['recommendations']['lifestyle_factors']:
        print("\n  Lifestyle and Preventive Measures:")
        for factor in result['recommendations']['lifestyle_factors'][:8]:  # Limit to first 8
            print(f"    • {factor}")
    
    if result['recommendations']['monitoring_parameters']:
        print("\n  Recommended Monitoring:")
        for monitor in result['recommendations']['monitoring_parameters'][:5]:  # Limit to first 5
            print(f"    • {monitor}")
    
    # Clinical Decision Support
    print("\nCLINICAL DECISION SUPPORT NOTES")
    print("-"*80)
    if result['probability'] >= 0.7:
        print("  • Further diagnostic evaluation strongly suggested")
        print("  • Consider cardiology consultation for comprehensive management")
        print("  • Evaluate need for preventive pharmacotherapy based on guidelines")
    elif result['probability'] >= 0.3:
        print("  • Additional risk assessment may be beneficial")
        print("  • Lifestyle interventions indicated regardless of further testing")
        print("  • Reassess in 3-6 months or sooner if symptoms develop")
    else:
        print("  • Continue current preventive health maintenance")
        print("  • Routine follow-up as per age and risk factor guidelines")
        print("  • Patient education on symptom recognition advised")
    
    # Guidelines Reference
    print("\nGUIDELINES REFERENCE")
    print("-"*80)
    print("  • Based on ACC/AHA cardiovascular risk assessment guidelines")
    print("  • Interpret results within clinical context")
    print("  • Individualize management based on patient preferences and values")
    
    # Footer
    print("\n" + "="*80)
    print("CLINICAL DECISION SUPPORT DISCLAIMER")
    print("-"*80)
    print("This report provides risk assessment based on machine learning analysis")
    print("and is intended as clinical decision support. Final clinical judgment")
    print("and treatment decisions remain with the treating healthcare provider.")
    print("="*80)

# ============================================
# TEST WITH YOUR PATIENT
# ============================================

# Your patient data
patient = {
    'Age': 58,
    'Sex': 1,
    'Chest Pain Type': 1,
    'Resting Blood Pressure': 145,
    'Cholesterol': 290,
    'Fasting Blood Sugar': 1,
    'Resting ECG': 1,
    'Max Heart Rate': 125,
    'Exercise-Induced Angina': 1,
    'ST Depression': 3.2,
    'Slope of ST Segment': 3,
    'Number of Major Vessels': 2,
    'Thalassemia': 7
}

# Get prediction with recommendations
result = predict_with_recommendations(patient)

# Print clinical report
print_clinical_report(patient, result)


CARDIOVASCULAR RISK ASSESSMENT REPORT
Assessment Date: 2026-06-10 12:02
Report ID: 20260610120220

PATIENT INFORMATION
--------------------------------------------------------------------------------
  Age: 58 years
  Sex: Male
  Clinical Presentation: Symptomatic

KEY CLINICAL PARAMETERS
--------------------------------------------------------------------------------
  • Total Cholesterol: 290 mg/dL
  • Resting Blood Pressure: 145 mmHg
  • Fasting Glucose: >120 mg/dL
  • Maximum Heart Rate: 125 bpm
  • ST Depression: 3.2 mm
  • Major Vessels Affected: 2

ASSESSMENT RESULT
--------------------------------------------------------------------------------
  Finding: POSITIVE for coronary artery disease indicators
  Probability: 90.00%
  Risk Stratification: HIGH

IDENTIFIED RISK FACTORS
--------------------------------------------------------------------------------
  1. Age >55 years
  2. High Cholesterol (>240 mg/dL)
  3. Elevated Blood Pressure (>140 mmHg)
  4. Impaired Fasting Glucos